### Build forward and reverse graph from resources for all resource types

In [ ]:
from collections import defaultdict
import json
import os
from pathlib import Path
import gzip

def find_references(obj, references=None):
    if references is None:
        references = []
    
    if isinstance(obj, dict):
        for key, value in obj.items():
            if key == "reference" and isinstance(value, str):
                references.append(value)
            else:
                find_references(value, references)
    elif isinstance(obj, list):
        for item in obj:
            find_references(item, references)
    
    return references

dirname = os.getcwd()
path = Path(os.path.join(dirname, 'mimic-iv-clinical-database-demo-on-fhir-2.0/mimic-fhir/'))
pathlist = list(path.glob('*.ndjson'))
forward_graph = defaultdict(set)
reverse_graph = defaultdict(set)

resources_map = defaultdict(lambda: defaultdict(set))

for path in pathlist:
    with open(path, 'r', encoding='utf-8') as f:
        filename = Path(path.stem).stem
        resource_name = filename.removeprefix("Mimic")
        for suffix in ("ICU", "ED", "Mix"):
            resource_name = resource_name.removesuffix(suffix)
        if resource_name not in resources_map:
            resources_map[resource_name] = defaultdict(set)
        print(f"Processing: {resource_name}")
        for line in f:
            resource = json.loads(line)
            resource_id = f"{resource_name}/{resource.get('id', '')}"
            resources_map[resource_name][resource_id] = resource
            
            references = find_references(resource)
            
            for ref in references:
                forward_graph[resource_id].add(ref)
            
            for ref in references:
                reverse_graph[ref].add(resource_id)

In [ ]:
from graph_builder import build_fhir_graphs

# Load FHIR resources and build graphs
forward_graph, reverse_graph, resources_map = build_fhir_graphs()

### Create Patient bundles from the graphs.
#### Iterate through the reverse graph, which already have direct references for a patient
#### Extend with the forward references for the indirect refs.

In [ ]:
def build_patient_bundle(patient_id, reverse_graph, forward_graph, resources_map):
    bundle_resources = set()
    bundle_resources.add(patient_id)  # Add the patient itself
    
    # Get all direct references to this patient
    direct_refs = reverse_graph.get(patient_id, set())
    bundle_resources.update(direct_refs)
    
    # For each direct reference, get their forward references (indirect refs)
    for ref_id in direct_refs:
        indirect_refs = forward_graph.get(ref_id, set())
        bundle_resources.update(indirect_refs)
    
    return bundle_resources

# Build bundles for all patients
patient_bundles = {}
patient_count = 0

# Find all patient resources
for patient_id, patient_resource in resources_map['Patient'].items():
    patient_count += 1
    bundle = build_patient_bundle(patient_id, reverse_graph, forward_graph, resources_map)
    patient_bundles[patient_id] = {
        'resources': bundle,
        'resource_objects': []
    }
    
    # Get the actual resource objects
    for resource_id in bundle:
        resource_type = resource_id.split('/')[0]
        if resource_type in resources_map and resource_id in resources_map[resource_type]:
            patient_bundles[patient_id]['resource_objects'].append(
                resources_map[resource_type][resource_id]
            )

print(f"Created {len(patient_bundles)} patient bundles")
print(f"Total patients: {patient_count}")

# Show statistics for first patient
first_patient_id = list(patient_bundles.keys())[0]
bundle = patient_bundles[first_patient_id]
print(f"\nFirst patient ({first_patient_id}):")
print(f"\nTotal resources in bundle: {len(bundle['resources'])}")
print(f"\nResource types: {sorted(set(r.split('/')[0] for r in bundle['resources']))}")


#### Persist Patient Bundle FHIR resources

In [ ]:
from pathlib import Path
import json

def persist_patient_bundles(patient_bundles, output_dir, resources_map):
    output_dir.mkdir(parents=True, exist_ok=True)
    
    for patient_id, bundle_data in patient_bundles.items():
        patient_uuid = patient_id.split("/")[1]
        resource_objects = bundle_data["resource_objects"]
        
        resource_objects.sort(
            key=lambda r: (
                0 if r["resourceType"] == "Patient" else 1,
                r["resourceType"]
            )
        )
        
        fhir_bundle = {
            "resourceType": "Bundle",
            "type": "collection",
            "entry": [
                {"resource": resource}
                for resource in resource_objects
            ]
        }

        output_file = output_dir / f"{patient_uuid}.json"

        with open(output_file, "w", encoding="utf-8") as f:
            json.dump(
                fhir_bundle,
                f,
                indent=2,
                ensure_ascii=False
            )
    
    return len(patient_bundles)

In [ ]:
# Persist all patient bundles
output_dir = Path("input/mimic_fhir_bundles")
persisted = persist_patient_bundles(patient_bundles, output_dir, resources_map)
print(f"Saved {persisted} bundles to {output_dir}")

In [ ]:
import random

# Get all encounters for each patient and determine which ones to split
patient_encounters = {}
for patient_id in resources_map['Patient'].keys():
    encounters = []
    for ref_id in reverse_graph.get(patient_id, set()):
        if ref_id.startswith('Encounter/'):
            encounters.append(ref_id)
    patient_encounters[patient_id] = encounters

# Build bundles for all patients with hospital split
patient_bundles_a = {}
patient_bundles_b = {}

for patient_id, patient_resource in resources_map['Patient'].items():
    # Get all resources for this patient
    all_resources = build_patient_bundle(patient_id, reverse_graph, forward_graph, resources_map)
    
    encounters = patient_encounters[patient_id]
    
    # Determine which encounters go to hospital_B
    encounters_to_hospital_b = set()
    if len(encounters) > 1:
        num_to_move = max(1, len(encounters) // 2)
        encounters_to_hospital_b = set(random.sample(encounters, num_to_move))
    
    # Find all resources that belong to hospital_B encounters
    hospital_b_resources = {patient_id}  # Always include patient
    
    for encounter_id in encounters_to_hospital_b:
        hospital_b_resources.add(encounter_id)
        # Add all resources that reference this encounter
        resources_ref_encounter = reverse_graph.get(encounter_id, set())
        hospital_b_resources.update(resources_ref_encounter)
        # Add forward references from those resources
        for resource_id in resources_ref_encounter:
            hospital_b_resources.update(forward_graph.get(resource_id, set()))
    
    # Hospital A gets all resources except those in hospital_B
    hospital_a_resources = all_resources - (hospital_b_resources - {patient_id})
    
    # Build hospital_A bundle
    patient_bundles_a[patient_id] = {
        'resources': hospital_a_resources,
        'resource_objects': []
    }
    for resource_id in hospital_a_resources:
        resource_type = resource_id.split('/')[0]
        if resource_type in resources_map and resource_id in resources_map[resource_type]:
            patient_bundles_a[patient_id]['resource_objects'].append(
                resources_map[resource_type][resource_id]
            )
    
    # Only create hospital_B bundle if there are resources to split
    if len(hospital_b_resources) > 1:
        patient_bundles_b[patient_id] = {
            'resources': hospital_b_resources,
            'resource_objects': []
        }
        for resource_id in hospital_b_resources:
            resource_type = resource_id.split('/')[0]
            if resource_type in resources_map and resource_id in resources_map[resource_type]:
                patient_bundles_b[patient_id]['resource_objects'].append(
                    resources_map[resource_type][resource_id]
                )

print(f"Created {len(patient_bundles_a)} patient bundles for hospital_A")
print(f"Created {len(patient_bundles_b)} patient bundles for hospital_B")
print(f"Patients with encounters split between hospitals: {len(patient_bundles_b)}")

# Persist bundles using the persist function
persisted_a = persist_patient_bundles(patient_bundles_a, Path("input/hospital_A"), resources_map)
persisted_b = persist_patient_bundles(patient_bundles_b, Path("input/hospital_B"), resources_map)

print(f"Saved {persisted_a} bundles to input/hospital_A")
print(f"Saved {persisted_b} bundles to input/hospital_B")

In [ ]:
from collections import Counter

service_type_counts = Counter()

for patient_id, encounters in patient_encounters.items():

    services = set()

    for encounter_id in encounters:

        encounter = resources_map["Encounter"][encounter_id]

        service = (
            encounter
            .get("serviceType", {})
            .get("coding", [{}])[0]
            .get("code")
        )

        if service:
            services.add(service)

    service_type_counts[len(services)] += 1

print(service_type_counts)

In [ ]:

from collections import Counter

def get_patient_locations(patient_id, reverse_graph, resources_map):
    locations = set()
    
    # Get all encounters for this patient
    for ref_id in reverse_graph.get(patient_id, set()):
        if ref_id.startswith('Encounter/'):
            encounter = resources_map['Encounter'].get(ref_id)
            if encounter:
                # Get location references from encounter
                location_refs = encounter.get('location', [])
                for loc_ref in location_refs:
                    location_id = loc_ref.get('location', {}).get('reference')
                    if location_id:
                        locations.add(location_id)
    
    return locations

location_counts = Counter()
for patient_id in resources_map['Patient'].keys():
    locations = get_patient_locations(patient_id, reverse_graph, resources_map)
    location_counts[len(locations)] += 1

print("Location counts (sorted by number of locations per patient):")
for num_locations in sorted(location_counts.keys()):
    print(f"  {num_locations} locations: {location_counts[num_locations]} patients")